### **Laboratorio 12: optimización adaptativa con SQuAD real**

**Objetivo:** Trabajar sobre los CSV reales, programar los análisis y entregar un notebook con celdas completadas. No volver a ejecutar el ranking ni entrenar un LLM.

#### **1. Preparación y verificación**

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
base = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'Semana12'/'retrieval_real.py').is_file()), None)
if base is None:
    raise FileNotFoundError('Abrir el cuaderno dentro de CC-0F4')
if str(base) not in sys.path:
    sys.path.insert(0, str(base))
from Semana12 import retrieval_real as real
print('Raíz del curso:', base)
if not (real.RESULTS/'resumen.json').is_file():
    raise FileNotFoundError('Ejecutar una vez en Docker: python -m Semana12.retrieval_real --run')
resumen = real.load_results()
test = pd.read_csv(real.RESULTS/'resultados_todas_las_consultas_test.csv')
calibracion = pd.read_csv(real.RESULTS/'calibracion_solo_desarrollo.csv')
assert len(test) == resumen['split']['test'] and test.question_id.is_unique
print('Resultados verificados, no se repite el ranking.')

#### **2. Implementar métricas y comparar baselines**

Implementa una función `compute_metrics(hit, context, lam)` que devuelva acierto, contexto y utilidad. No usar `real.summarize` para tu implementación. Construye una tabla para los cinco k fijos y B a lambda=0.025. Contrasta con el resumen y explica por qué top-3 y top-4 son controles necesarios. Comprueba las desigualdades por pregunta.

In [ ]:
# Completar: implementar compute_metrics y generar seis filas.
# Entradas: test.hit_top1 ... test.hit_top5, test.hit_B_adaptive, test.context_B.
# Comprobar con np.isclose contra resumen['test_results'].
print('Políticas que se deben comparar:', list(resumen['test_results']))

#### **3. Presupuesto equivalente y sensibilidad a lambda**

Calcula el costo medio de B y la mezcla esperada de los dos k adyacentes sin consultar aciertos para elegir su peso. Implementa la fórmula de interpolación y contrástala con el resumen. Dibuja U para las políticas congeladas y B recalibrada.

 Obtén los intervalos de lambda donde cada política congelada gana (incluye top-2 a top-4). Discute empates y explica por qué no debes usar esa frontera para calibrar con test.

In [ ]:
curvas = pd.read_csv(real.RESULTS/'sensibilidad_lambda.csv')
# Completar: calcular mezcla a igual costo y graficar curvas.
# Comparar con la frontera exacta registrada, sin elegir por test.
display(pd.DataFrame(resumen['lambda_frontier_frozen_test']))

#### **4. Calibración y estabilidad por semillas**

Implementa la selección del umbral con desempate por menor valor usando la tabla de desarrollo. Calcula la diferencia entre los dos mejores. Con las ocho semillas fijadas, reporta frecuencia de umbrales y rango de acierto, contexto y utilidad de B. Compara con la semilla principal sin escoger la más favorable. Explica por qué las particiones no son independientes.

In [ ]:
semillas = pd.read_csv(real.RESULTS/'sensibilidad_semillas.csv')
display(calibracion)
# Completar: ordenar, seleccionar, contar frecuencias, calcular rangos y graficar.

#### **5. Bootstrap emparejado por artículo**

Implementa `cluster_ci(frame, diff, repeats=2000, seed=2026)`. Agrupa la diferencia por artículo, guarda suma y tamaño, remuestrea artículos completos y divide por el total de preguntas de cada réplica. Calcula IC95 percentil de B-A0 en acierto y B-F en utilidad, donde F es el fijo seleccionado en desarrollo. Compara con el resumen. 

Calcula además el promedio macro de las diferencias. Explica por qué el intervalo micro no debe acompañar un estimando macro y por qué el análisis es condicional a políticas ajustadas.

In [ ]:
fijo = resumen['best_fixed_on_development']
print('Baseline fijo seleccionado:', fijo)
# Completar: construir diferencias por pregunta, agrupar, remuestrear y contrastar.
# Se puede usar el mismo generador np.random.default_rng(2026).

#### **6. Diagnóstico de errores y heterogeneidad**

Reproduce la muestra de tres errores: elige artículos distintos al azar con semilla 2026 y una pregunta por artículo, si existen suficientes. Consulta el texto del párrafo original y el top-1 recuperado. Analiza dos errores de ranking y dos casos en los que B pierde un acierto de A1. Produce tablas por artículo y palabra interrogativa inicial, incluyendo tamaños de grupo y número de fallos top-5. Identifica los grupos con menor acierto sin generalizar grupos pequeños. Contrasta el desglose generado con tu propio cálculo.

In [ ]:
docs, questions, provenance = real.load_squad()
errores = test[test.hit_top5 == 0].copy()
perdidas = test[(test.hit_top5 == 1) & (test.hit_B_adaptive == 0)]
muestra = pd.read_csv(real.RESULTS/'muestra_errores_test.csv')
display(muestra[['question_id','article','question','paragraph_id','top1_paragraph_id']])
# Completar: reproducir muestreo, inspeccionar textos con docs, desglosar y argumentar.

#### **7. Entrega y defensa**

Entregar notebook con código propio, resultados reproducibles, tabla de seis políticas, análisis a igual presupuesto, curva de lambda, estabilidad de umbral, dos IC95, análisis de cuatro errores y desglose. Adjuntar referencia al resumen y sus hashes. Explicar limitaciones del qrel, del costo por párrafo, del test ya observado y de la inferencia estadística. No afirmar eficacia de LLM, LoRA ni test-time reasoning.

Criterios: métricas y baselines 20 %, presupuesto y frontera 20 %, calibración 15 %, bootstrap 20 %, diagnóstico 15 %, defensa y reproducibilidad 10 %.